<div dir="rtl" align="right">

# مصفوفةُ الارتباطِ بينَ القنواتِ

**مجموعةُ البياناتِ**: BNCI2014-001 (تَخيّلٌ حركيٌّ)
**المُشاركُ**: 1
**النموذجُ**: MotorImagery (n_classes=2)
**القنواتُ**: 22 EEG
**معدّلُ أخذِ العيناتِ**: 250 Hz

---

## نظرةٌ عامّةٌ

هذا الدفترُ يَحسبُ مصفوفةَ ارتباطِ بيرسون المُتوسطةَ عبرَ جميعِ قنواتِ EEG الـ 22 في BNCI2014-001، مُتوسطةً على جميعِ المحاولاتِ، ويُصوّرُها كَخريطةِ حرارةٍ تفاعليّةٍ.

## ماذا يَفعلُ هذا الدفترُ

- يحمّلُ BNCI2014-001 لِلمُشاركِ 1 عبرَ MOABB
- يَستخرجُ الحقبَ بِنموذجِ MotorImagery
- يَحسبُ مصفوفاتِ الارتباطِ لِكلِّ محاولةٍ بِاستخدامِ np.corrcoef
- يَتوسطُ على جميعِ المحاولاتِ ويَرسمُ النتيجةَ كَخريطةِ حرارةٍ

## المُخرجاتُ المُتوقّعةُ

- مصفوفةُ ارتباطٍ 22x22 بِقيمٍ من -1 إلى +1
- ارتباطٌ عالٍ على القُطرِ (كلُّ قناةٍ معَ نفسِها)
- القنواتُ المُتجاورةُ (مثلُ C3-C1, Cz-C2) تُظهرُ ارتباطاً أعلى
- قنواتُ EEG الخاليةُ من EOG تَكشفُ أنماطَ الاتصالِ المكانيِّ

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| dataset | BNCI2014_001 | مجموعةُ بياناتِ تَخيّلٍ حركيٍّ من MOABB |
| subjects | [1] | المُشاركُ 1 فقط |
| n_classes | 2 | فئاتُ نموذجِ MotorImagery |
| method | Pearson | نوعُ الارتباطِ |
| n_channels | 22 | قنواتُ EEG في الحقبِ |


</div>


<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>


In [ ]:
!pip install moabb mne scipy numpy plotly


<div dir="rtl" align="right">

## 2. تحميلُ مجموعةِ بياناتِ MOABB

MOABB تُنزّلُ البياناتِ تلقائياً عندَ أولِ استخدامٍ (~44 ميجابايت لِلمُشاركِ 1). التشغيلاتُ اللاحقةُ تَستخدمُ البياناتِ المُخزّنةَ.


</div>


In [ ]:
from moabb.datasets import BNCI2014_001
ds = BNCI2014_001()
sessions = ds.get_data(subjects=[1])
subject_key = list(sessions.keys())[0]
session_dict = sessions[subject_key]
n_sessions = len(session_dict)
n_runs = len(next(iter(session_dict.values())))
first_run = next(iter(next(iter(session_dict.values())).values()))
n_channels_raw = len(first_run.ch_names)
sfreq = first_run.info['sfreq']
print(f'Subject 1: {n_sessions} sessions, {n_runs} runs/session')
print(f'Raw channels: {n_channels_raw}, Sampling rate: {sfreq} Hz')
print(f'Channel names: {first_run.ch_names}')



In [ ]:
from moabb.paradigms import MotorImagery
paradigm = MotorImagery(n_classes=2)
X, labels, meta = paradigm.get_data(dataset=ds, subjects=[1])
print(f'X shape: {X.shape}  (n_trials, n_channels, n_samples)')
print(f'Labels shape: {labels.shape}')
print(f'Meta shape: {meta.shape}')



<div dir="rtl" align="right">

## 3. استكشافُ البياناتِ

نَطبعُ أشكالَ الحقبِ ومعلوماتِ القنواتِ.

</div>


In [ ]:
import numpy as np
unique_labels, counts = np.unique(labels, return_counts=True)
print(f'Epoch channels: {X.shape[1]}')
print(f'Epoch samples: {X.shape[2]}')
print(f'Epoch duration: {X.shape[2] / sfreq:.2f} s')
print(f'Unique labels: {list(unique_labels)}')
print(f'Trials per class: {dict(zip(unique_labels, counts))}')
print(f'Total trials: {X.shape[0]}')



<div dir="rtl" align="right">

## 4. تطبيقُ التحليلِ

نَحسبُ مصفوفةَ الارتباطِ لِكلِّ محاولةٍ ونُتوسطُ على جميعِ المحاولاتِ.


</div>


In [ ]:
import numpy as np
n_trials, n_channels, n_samples = X.shape
corr_sum = np.zeros((n_channels, n_channels))
for trial in range(n_trials):
    trial_data = X[trial]
    std = np.std(trial_data, axis=1, keepdims=True)
    std[std == 0] = 1.0
    normed = (trial_data - np.mean(trial_data, axis=1, keepdims=True)) / std
    corr_sum += np.corrcoef(normed)
corr_matrix = corr_sum / n_trials
print(f'Correlation matrix shape: {corr_matrix.shape}')
print(f'Mean off-diagonal correlation: {np.mean(corr_matrix[~np.eye(n_channels, dtype=bool)]):.3f}')



<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- القُطرُ مُرتبطٌ تماماً (القيمةُ = 1.0)
- الأقطابُ المُتجاورةُ تُظهرُ ارتباطاً أعلى (ألوانٌ دافئةٌ)
- الأقطابُ البعيدةُ تُظهرُ ارتباطاً أقلَّ أو سالباً (ألوانٌ باردةٌ)
- المصفوفةُ مُتماثلةٌ حولَ القُطرِ



</div>


In [ ]:
import plotly.graph_objects as go
raw_ch = first_run.ch_names
eog_idx = [i for i, n in enumerate(raw_ch) if n.startswith('EOG') or n == 'STI']
ch_labels = [n for i, n in enumerate(raw_ch) if i not in eog_idx]
fig = go.Figure(data=go.Heatmap(z=corr_matrix, x=ch_labels, y=ch_labels,
                                 colorscale='RdBu', zmin=-1, zmax=1))
fig.update_layout(title='Channel Correlation Matrix - BNCI2014-001',
                  xaxis_title='Channel', yaxis_title='Channel',
                  width=700, height=700)
fig.show()



<div dir="rtl" align="right">

## خلاصةٌ

- مصفوفةُ الارتباطِ تَكشفُ العلاقاتِ المكانيةَ بينَ قنواتِ EEG
- الأقطابُ المُتجاورةُ أكثرُ ارتباطاً منَ البعيدةِ
- التوسيطُ على المحاولاتِ يُعطي تقديراً مُستقرّاً لِاتصالِ القنواتِ
- هذه المصفوفةُ مفيدةٌ لِاختيارِ الميزاتِ وتقليلِ القنواتِ



</div>
